# Trimmed csv file generation from raw file 

In [17]:
import os
import glob
import pandas as pd
import numpy as np
import re

In [22]:
# base dir- dltdv8 xyz csv files lie here. output dir- where the trimmed csvs should be saved
BASE_DIR = r"../../Tanvi-files/DataForAnalysis"
INPUT_PATTERN = "**/DLTdv8_data_*xyzpts*.csv"

OUTPUT_DIR = r"../../Tanvi-files/xyz_Raw_Trimmed"
OUTPUT_SUFFIX = "_trimmed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# FIND FILES
# ==========================================================

csv_files = sorted(glob.glob(os.path.join(BASE_DIR, INPUT_PATTERN), recursive=True))
if not csv_files:
    raise FileNotFoundError("No DLTdv8 CSV files found.")

print(f"\nFound {len(csv_files)} DLTdv8 CSV files\n")

# ==========================================================
# TRIM + CENTER + DROP pt>7
# ==========================================================

for csv_path in csv_files:
    fname = os.path.basename(csv_path)
    trial_name = "_".join(os.path.basename(csv_path).split('_')[2:4])

    print("=" * 60)
    print(f"Processing trial: {trial_name}")
    print(f"File: {fname}")

    # Load CSV WITH headers
    df = pd.read_csv(csv_path)
    n_frames = len(df)

    print(f"Available frames: 0 → {n_frames - 1}")

    # ------------------------------------------------------
    # Ask per-trial frame range
    # ------------------------------------------------------
    while True:
        try:
            start_frame = int(input("Enter START frame (inclusive): "))
            end_frame   = int(input("Enter END frame (inclusive): "))

            if start_frame < 0 or end_frame >= n_frames:
                raise ValueError("Frame out of bounds")

            if start_frame >= end_frame:
                raise ValueError("START must be < END")

            break

        except ValueError as e:
            print(f" {e}. Try again.\n")

    # ------------------------------------------------------
    # Trim frames
    # ------------------------------------------------------
    df_trimmed = df.iloc[start_frame:end_frame + 1].copy()

    # Add frame column
    df_trimmed.insert(
        0,
        "frame",
        np.arange(start_frame, end_frame + 1)
    )

    # ------------------------------------------------------
    # COMPUTE CENTER (pt3 & pt4)
    # ------------------------------------------------------
    required_cols = ["pt3_X", "pt3_Y", "pt3_Z", "pt4_X", "pt4_Y", "pt4_Z"]
    missing = [c for c in required_cols if c not in df_trimmed.columns]
    if missing:
        raise ValueError(f"Missing required columns for center computation: {missing}")

    df_trimmed["center_X"] = 0.5 * (df_trimmed["pt3_X"] + df_trimmed["pt4_X"])
    df_trimmed["center_Y"] = 0.5 * (df_trimmed["pt3_Y"] + df_trimmed["pt4_Y"])
    df_trimmed["center_Z"] = 0.5 * (df_trimmed["pt3_Z"] + df_trimmed["pt4_Z"])

    # ------------------------------------------------------
    # DROP points AFTER pt7 (pt8, pt9, ...)
    # ------------------------------------------------------
    keep_cols = ["frame"]

    for col in df_trimmed.columns:
        m = re.match(r"pt(\d+)_[XYZ]", col)
        if m and int(m.group(1)) <= 7:
            keep_cols.append(col)

    # Always keep center
    keep_cols += ["center_X", "center_Y", "center_Z"]

    df_trimmed = df_trimmed[keep_cols]

    # ------------------------------------------------------
    # Save
    # ------------------------------------------------------
    out_name = f"{trial_name.replace('.csv', OUTPUT_SUFFIX + '.csv')}"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    df_trimmed.to_csv(out_path, index=False)

    print(f" Saved → {out_name}")
    print(f"Frames: {start_frame} → {end_frame}\n")

print("All trials trimmed, center computed, pt>7 removed, and saved successfully.")


Found 4 DLTdv8 CSV files

Processing trial: Trial2_200rpmxyzpts.csv
File: DLTdv8_data_Trial2_200rpmxyzpts.csv
Available frames: 0 → 889
Enter START frame (inclusive): 50
Enter END frame (inclusive): 285
 Saved → Trial2_200rpmxyzpts_trimmed.csv
Frames: 50 → 285

Processing trial: Trial3_180rpmxyzpts.csv
File: DLTdv8_data_Trial3_180rpmxyzpts.csv
Available frames: 0 → 439
Enter START frame (inclusive): 15
Enter END frame (inclusive): 250
 Saved → Trial3_180rpmxyzpts_trimmed.csv
Frames: 15 → 250

Processing trial: Trial4_400rpmxyzpts.csv
File: DLTdv8_data_Trial4_400rpmxyzpts.csv
Available frames: 0 → 755
Enter START frame (inclusive): 80
Enter END frame (inclusive): 525
 Saved → Trial4_400rpmxyzpts_trimmed.csv
Frames: 80 → 525

Processing trial: Trial5_180rpmxyzpts.csv
File: DLTdv8_data_Trial5_180rpmxyzpts.csv
Available frames: 0 → 1118
Enter START frame (inclusive): 0
Enter END frame (inclusive): 650
 Saved → Trial5_180rpmxyzpts_trimmed.csv
Frames: 0 → 650

All trials trimmed, center com